<a href="https://colab.research.google.com/github/junnong15/20231830-/blob/master/Stock_Trading_ko_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 5.0 MB/s eta 0:00:00


In [ ]:
!pip install finance-datareader --upgrade -q
import FinanceDataReader as fdr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 1.9 MB/s eta 0:00:00


In [ ]:
#1️⃣ Google Drive 마운트 및 폴더 준비
from google.colab import drive
import os

# 📂 Google Drive 마운트
drive.mount('/content/drive')

# ✅ 저장 경로 지정
SAVE_DIR = '/content/drive/MyDrive/StockSigma'
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

print(f"✅ 파일 저장 경로: {SAVE_DIR}")

Mounted at /content/drive
✅ 파일 저장 경로: /content/drive/MyDrive/StockSigma


In [ ]:
# 2️⃣ 한글 폰트 설정 (Colab 또는 Windows 자동 판별)
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

if platform.system() != 'Windows':
    !apt-get -qq install -y fonts-nanum > /dev/null
    font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
    if os.path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.family'] = 'NanumGothic'
    else:
        print("❌ NanumGothic 폰트를 찾을 수 없습니다.")
else:
    plt.rcParams['font.family'] = 'Malgun Gothic'

plt.rcParams['axes.unicode_minus'] = False

In [ ]:
#3️⃣ 라이브러리 불러오기 및 종목코드 변환 함수 정의
def resolve_korean_symbol(user_input):
    """한글 종목명 → 종목코드.KS, 숫자 종목코드에도 .KS 자동"""
    if user_input.isdigit():
        return user_input + '.KS'
    if any(ord(c) > 127 for c in user_input):
        try:
            krx = fdr.StockListing('KRX')
            match = krx[krx['Name'] == user_input]
            if not match.empty:
                code = match['Code'].iloc[0]  # ✅ 안전한 단일값 접근
                return code + '.KS'
        except Exception as e:
            print(f"❌ 종목 변환 오류: {e}")
    return user_input

In [ ]:
print(resolve_korean_symbol("삼성전자우"))  # ✅ 005935.KS 나와야 함
print(resolve_korean_symbol("카카오"))      # ✅ 035720.KS
print(resolve_korean_symbol("AAPL"))       # ✅ AAPL


005935.KS
035720.KS
AAPL


In [ ]:
# 4️⃣ 분석 함수 정의
def analyze_stock_sigma(ticker, months, window=20):
    # datetime, timedelta, pandas, and yfinance need to be imported
    from datetime import datetime, timedelta
    import pandas as pd
    import yfinance as yf
    import os
    import matplotlib.pyplot as plt # Needed for plotting

    today_str = datetime.today().strftime("%Y%m%d")
    period_str = f"{months}M"
    end = datetime.today()
    start = end - timedelta(days=months * 30)

    print(f"\n[분석 대상: {ticker}] (최근 {months}개월)\n")

    try:
        df = yf.download(ticker, start=start, end=end)
    except Exception as e:
        print(f"❌ yfinance 다운로드 오류: {e}")
        return

    if df.empty:
        print("❌ 주가 데이터를 찾을 수 없습니다.")
        return

    df['MA'] = df['Close'].rolling(window).mean()
    df['STD'] = df['Close'].rolling(window).std()
    df['+1σ'] = df['MA'] + df['STD']
    df['-1σ'] = df['MA'] - df['STD']
    df['+2σ'] = df['MA'] + 2 * df['STD']
    df['-2σ'] = df['MA'] - 2 * df['STD']
    df = df.dropna()

    if df.empty:
        print("❌ 유효한 데이터 부족 (이동평균 계산 후 데이터 없음)")
        return

    # ✅ 최신값 추출 (.iloc[-1] 사용 또는 그냥 [-1])
    # Changed .iat[-1] to .iloc[-1] for correct Series indexing
    # Use .item() to get the scalar value from the Series
    price   = df['Close'].iloc[-1].item()
    ma      = df['MA'].iloc[-1].item()
    std     = df['STD'].iloc[-1].item()
    upper1  = df['+1σ'].iloc[-1].item()
    lower1  = df['-1σ'].iloc[-1].item()
    upper2  = df['+2σ'].iloc[-1].item()
    lower2  = df['-2σ'].iloc[-1].item()

    if pd.isna(upper2) or pd.isna(lower2):
        signal = "❗ 시그마 계산 불가"
    elif price > upper2:
        signal = "📈 과매수 → 매도 고려"
    elif price < lower2:
        signal = "📉 과매도 → 매수 고려"
    else:
        signal = "🤝 정상 범위 → 관망 또는 보유"

    print(f"📊 {ticker} 분석 결과")
    print(f"현재가: {price:.2f}")
    print(f"20일 MA: {ma:.2f} / +2σ: {upper2:.2f} / -2σ: {lower2:.2f}")
    print(f"👉 추천 판단: {signal}")
    print("\n🕵️ 최근 5일 데이터:")
    print(df.tail())

    # SAVE_DIR is defined outside this function but is used here.
    # Assuming it's globally available or passed in a real scenario.
    # In this Colab environment, it's defined in a previous cell.
    base = f"{today_str}_{ticker}_{period_str}"
    image_path = os.path.join(SAVE_DIR, f"{base}.png")
    excel_path = os.path.join(SAVE_DIR, f"{base}.xlsx")

    # ✅ 차트 저장 및 시각화
    plt.figure(figsize=(14, 6))
    plt.plot(df.index, df['Close'], label='종가', linewidth=1.5)
    plt.plot(df.index, df['MA'], label='20일 이동평균', linestyle='--')
    plt.plot(df.index, df['+1σ'], label='+1σ', linestyle=':')
    plt.plot(df.index, df['-1σ'], label='-1σ', linestyle=':')
    plt.plot(df.index, df['+2σ'], label='+2σ', linestyle='--')
    plt.plot(df.index, df['-2σ'], label='-2σ', linestyle='--')
    plt.title(f"{ticker} 주가 및 시그마 밴드 (최근 {months}개월)")
    plt.xlabel("날짜"); plt.ylabel("가격")
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig(image_path)
    print(f"✅ 차트 이미지 저장 완료: {image_path}")
    # plt.show() # Removed plt.show() here to avoid displaying plots automatically

    summary = pd.DataFrame([{
        'Ticker': ticker,
        'Date': df.index[-1].strftime('%Y-%m-%d'),
        'Close': price,
        'MA': ma,
        'STD': std,
        '+1σ': upper1,
        '-1σ': lower1,
        '+2σ': upper2,
        '-2σ': lower2,
        'Signal': signal
    }])

    # xlsxwriter needs to be imported or available in the environment
    # The !pip install xlsxwriter cell should handle this.
    # However, explicitly importing it here might be safer if running this function stand-alone.
    # import xlsxwriter # Uncomment if needed

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        df.to_excel(writer, sheet_name='Daily Data')
        summary.to_excel(writer, sheet_name='Summary', index=False)
        # Ensure the image exists before trying to insert it
        if os.path.exists(image_path):
             writer.sheets['Summary'].insert_image('K2', image_path, {'x_scale': 0.9, 'y_scale': 0.9})
        else:
             print(f"❌ Warning: Image file not found at {image_path}. Cannot insert into Excel.")

    print(f"✅ 엑셀 저장 완료: {excel_path}")

    # Display the plot after saving to Excel
    plt.show()

In [ ]:


1#5️⃣ 사용자 입력 & 실행
# 사용자로부터 한국 주식 코드(예: 005930) 또는 이름(예: 삼성전자) 입력받아 분석
user_input = input("▶ 종목코드 또는 종목명 입력 (예: 삼성전자, 005930, AAPL): ").strip()
ticker = resolve_korean_symbol(user_input)
months = int(input("▶ 분석 기간 (1/3/6/12개월): ").strip() or 12)
analyze_stock_sigma(ticker, months)
